# 07 - Model Training

Objective

This notebook prepares the modeling datasets, trains baseline and tuned candidate models, and selects the candidate final model for downstream evaluation.

The workflow preserves temporal ordering, prevents target leakage, compares Logistic Regression, Random Forest, and XGBoost, and persists modeling checkpoints for the evaluation and explainability notebooks.

Holdout testing, calibration analysis, fitted-model artifact saving, and model explainability are performed in Notebooks 08 and 09.

#### Load project configuration


In [0]:
import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

# Load the project configuration

from config import project_config as cfg

print("Project configuration loaded successfully.")


#### Load and validate the feature dataset

The model-training process begins by loading the managed `flights_features` Delta table produced by the Feature Engineering notebook.

Before splitting or modelling, the dataset is validated to confirm that:

- The required Unity Catalog table exists
- The target variable is available
- The flight date is stored as a valid date
- All required schedule-time predictors are present
- The dataset contains records suitable for chronological splitting


In [0]:
from __future__ import annotations

import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

from config import project_config as cfg
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T



FEATURE_TABLE = cfg.FEATURES_TABLE
TARGET_COLUMN = cfg.TARGET_COLUMN
DATE_COLUMN = cfg.FLIGHT_DATE_COLUMN


def require_table(table_name: str) -> None:
    """Raise an error when a required Unity Catalog table is unavailable."""
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run the feature-engineering notebook (06) before continuing."
        )


require_table(FEATURE_TABLE)

df_features: DataFrame = spark.table(FEATURE_TABLE)

required_columns = cfg.MODEL_TRAINING_REQUIRED_COLUMNS

missing_columns = sorted(required_columns - set(df_features.columns))

if missing_columns:
    raise ValueError(
        "Model-training validation failed. "
        f"Missing required columns: {missing_columns}"
    )

feature_row_count = df_features.count()
feature_column_count = len(df_features.columns)

date_type = df_features.schema[DATE_COLUMN].dataType

if not isinstance(date_type, T.DateType):
    raise TypeError(
        f"{DATE_COLUMN} must be a Spark date column, "
        f"but found {date_type.simpleString()}."
    )

print("Feature dataset loaded and validated successfully.")
print(f"Source table: {FEATURE_TABLE}")
print(f"Total records: {feature_row_count:,}")
print(f"Total columns: {feature_column_count}")
print(f"Prediction target: {TARGET_COLUMN}")
print(f"Date column type: {date_type.simpleString()}")

#### Date range and target distribution

Before defining the chronological training, validation, and test periods, the feature dataset is examined to confirm its available date range and the distribution of the binary target variable.

This review supports two important modelling decisions:

- Selecting non-overlapping chronological split periods
- Assessing whether the delayed and on-time classes are imbalanced

No records are modified during this analysis.


In [0]:
dataset_profile = (
    df_features
    .select(
        F.min("FL_DATE").alias("MIN_FL_DATE"),
        F.max("FL_DATE").alias("MAX_FL_DATE"),
        F.count("*").alias("TOTAL_RECORDS"),
        F.sum(
            F.when(F.col("ARR_DEL15") == 0, 1).otherwise(0)
        ).alias("ON_TIME_RECORDS"),
        F.sum(
            F.when(F.col("ARR_DEL15") == 1, 1).otherwise(0)
        ).alias("DELAYED_RECORDS"),
    )
    .withColumn(
        "ON_TIME_PERCENTAGE",
        F.round(
            F.col("ON_TIME_RECORDS") / F.col("TOTAL_RECORDS") * 100,
            4,
        ),
    )
    .withColumn(
        "DELAYED_PERCENTAGE",
        F.round(
            F.col("DELAYED_RECORDS") / F.col("TOTAL_RECORDS") * 100,
            4,
        ),
    )
)

display(dataset_profile)

#### Create chronological train, validation, and test splits

The dataset is divided chronologically rather than randomly because the model is intended to predict future flight-delay risk from historical observations.

The split periods are defined as follows:

- **Training period:** January 1, 2025 to August 31, 2025
- **Validation period:** September 1, 2025 to October 31, 2025
- **Test period:** November 1, 2025 to December 31, 2025

This design ensures that later flight outcomes are not used to train models evaluated on earlier periods. The validation dataset will support model and hyperparameter selection, while the test dataset will remain untouched until final evaluation.


In [0]:
TRAIN_END_DATE = cfg.TRAIN_END_DATE
VALIDATION_START_DATE = cfg.VALIDATION_START_DATE
VALIDATION_END_DATE = cfg.VALIDATION_END_DATE
TEST_START_DATE = cfg.TEST_START_DATE

df_train = df_features.filter(
    F.col("FL_DATE") <= F.to_date(F.lit(TRAIN_END_DATE))
)

df_validation = df_features.filter(
    (F.col("FL_DATE") >= F.to_date(F.lit(VALIDATION_START_DATE)))
    & (F.col("FL_DATE") <= F.to_date(F.lit(VALIDATION_END_DATE)))
)

df_test = df_features.filter(
    F.col("FL_DATE") >= F.to_date(F.lit(TEST_START_DATE))
)

split_summary = (
    df_train.select(
        F.lit("TRAIN").alias("DATASET"),
        F.min("FL_DATE").alias("MIN_DATE"),
        F.max("FL_DATE").alias("MAX_DATE"),
        F.count("*").alias("TOTAL_RECORDS"),
        F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
    )
    .unionByName(
        df_validation.select(
            F.lit("VALIDATION").alias("DATASET"),
            F.min("FL_DATE").alias("MIN_DATE"),
            F.max("FL_DATE").alias("MAX_DATE"),
            F.count("*").alias("TOTAL_RECORDS"),
            F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
        )
    )
    .unionByName(
        df_test.select(
            F.lit("TEST").alias("DATASET"),
            F.min("FL_DATE").alias("MIN_DATE"),
            F.max("FL_DATE").alias("MAX_DATE"),
            F.count("*").alias("TOTAL_RECORDS"),
            F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
        )
    )
    .withColumn(
        "DELAY_PERCENTAGE",
        F.round(F.col("DELAY_RATE") * 100, 4),
    )
    .drop("DELAY_RATE")
)

display(split_summary)

#### Split validation summary

The chronological split produced three non-overlapping datasets whose combined record count matches the complete feature dataset.

The target distribution varies across the periods:

- The training period has a delay rate of approximately 22.91%.
- The validation period has a lower delay rate of approximately 18.55%.
- The test period has a higher delay rate of approximately 23.71%.

This variation reflects temporal changes in airline operations and confirms the importance of evaluating the model on future periods rather than using a random split.


#### Engineer Strictly Leakage-Safe Historical Features

Historical delay-rate features summarize prior airline, airport, and route performance. For every training date, both the entity history and its smoothing prior are calculated from **strictly earlier dates only**. Outcomes from the current date and all later dates are excluded.

The initial value `0.5` is used only when the dataset contains no earlier observations. Once earlier flights exist, the cumulative delay rate through the preceding date becomes the date-specific smoothing prior. This removes the former January–August global fallback that allowed early flights to indirectly receive information from later months.


In [0]:
from pyspark.sql.window import Window

SMOOTHING_STRENGTH = 100.0
INITIAL_DELAY_PRIOR = 0.5

# A causal global prior for each date, based only on previous dates.
global_daily_stats = (
    df_train
    .groupBy("FL_DATE")
    .agg(
        F.count("*").alias("GLOBAL_DAILY_FLIGHTS"),
        F.sum(F.col(TARGET_COLUMN).cast("long")).alias(
            "GLOBAL_DAILY_DELAYS"
        ),
    )
)

global_history_window = (
    Window
    .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
    .rowsBetween(Window.unboundedPreceding, -1)
)

date_specific_priors = (
    global_daily_stats
    .withColumn(
        "GLOBAL_PRIOR_FLIGHTS",
        F.sum("GLOBAL_DAILY_FLIGHTS").over(global_history_window),
    )
    .withColumn(
        "GLOBAL_PRIOR_DELAYS",
        F.sum("GLOBAL_DAILY_DELAYS").over(global_history_window),
    )
    .withColumn(
        "DATE_PRIOR_DELAY_RATE",
        F.when(
            F.col("GLOBAL_PRIOR_FLIGHTS").isNull()
            | (F.col("GLOBAL_PRIOR_FLIGHTS") == 0),
            F.lit(INITIAL_DELAY_PRIOR),
        ).otherwise(
            F.col("GLOBAL_PRIOR_DELAYS").cast("double")
            / F.col("GLOBAL_PRIOR_FLIGHTS").cast("double")
        ),
    )
    .select("FL_DATE", "DATE_PRIOR_DELAY_RATE")
)


def causal_entity_history(
    source_df,
    entity_columns,
    output_prefix,
):
    # Create a smoothed entity rate using only earlier dates.
    daily_flights = f"{output_prefix}_DAILY_FLIGHTS"
    daily_delays = f"{output_prefix}_DAILY_DELAYS"
    prior_flights = f"{output_prefix}_PRIOR_FLIGHTS"
    prior_delays = f"{output_prefix}_PRIOR_DELAYS"
    rate_column = f"{output_prefix}_HIST_DELAY_RATE"

    daily = (
        source_df
        .groupBy(*entity_columns, "FL_DATE")
        .agg(
            F.count("*").alias(daily_flights),
            F.sum(F.col(TARGET_COLUMN).cast("long")).alias(daily_delays),
        )
    )
    history_window = (
        Window
        .partitionBy(*entity_columns)
        .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
        .rowsBetween(Window.unboundedPreceding, -1)
    )

    return (
        daily
        .withColumn(prior_flights, F.sum(daily_flights).over(history_window))
        .withColumn(prior_delays, F.sum(daily_delays).over(history_window))
        .join(date_specific_priors, on="FL_DATE", how="left")
        .withColumn(
            rate_column,
            (
                F.coalesce(F.col(prior_delays).cast("double"), F.lit(0.0))
                + F.lit(SMOOTHING_STRENGTH)
                * F.col("DATE_PRIOR_DELAY_RATE")
            )
            /
            (
                F.coalesce(F.col(prior_flights).cast("double"), F.lit(0.0))
                + F.lit(SMOOTHING_STRENGTH)
            ),
        )
        .select(
            *entity_columns,
            "FL_DATE",
            prior_flights,
            prior_delays,
            rate_column,
        )
    )

print("Strictly causal date-specific smoothing priors created.")


#### Historical Airline Delay Rate

`AIRLINE_HIST_DELAY_RATE` uses the airline's outcomes from earlier dates and the causal date-specific prior. The current date and later months cannot influence the value.


In [0]:
airline_history_features = causal_entity_history(
    df_train,
    ["OP_UNIQUE_CARRIER"],
    "AIRLINE",
)

df_train_hist = (
    df_train
    .join(
        airline_history_features,
        on=["OP_UNIQUE_CARRIER", "FL_DATE"],
        how="left",
    )
    .join(date_specific_priors, on="FL_DATE", how="left")
    .withColumn(
        "AIRLINE_HIST_DELAY_RATE",
        F.coalesce(
            F.col("AIRLINE_HIST_DELAY_RATE"),
            F.col("DATE_PRIOR_DELAY_RATE"),
            F.lit(INITIAL_DELAY_PRIOR),
        ),
    )
    .drop("DATE_PRIOR_DELAY_RATE")
)

print(f"Training rows after causal airline join: {df_train_hist.count():,}")


#### Historical Origin-Airport Delay Rate

`ORIGIN_HIST_DELAY_RATE` uses only flights from earlier dates at the same origin, smoothed toward the cumulative delay rate available before the current date.


In [0]:
origin_history_features = causal_entity_history(
    df_train,
    ["ORIGIN"],
    "ORIGIN",
)

df_train_hist = (
    df_train_hist
    .join(origin_history_features, on=["ORIGIN", "FL_DATE"], how="left")
    .join(date_specific_priors, on="FL_DATE", how="left")
    .withColumn(
        "ORIGIN_HIST_DELAY_RATE",
        F.coalesce(
            F.col("ORIGIN_HIST_DELAY_RATE"),
            F.col("DATE_PRIOR_DELAY_RATE"),
            F.lit(INITIAL_DELAY_PRIOR),
        ),
    )
    .drop("DATE_PRIOR_DELAY_RATE")
)


#### Historical Destination-Airport Delay Rate

`DEST_HIST_DELAY_RATE` uses only flights from earlier dates with the same destination, with the same strictly causal smoothing rule.


In [0]:
dest_history_features = causal_entity_history(
    df_train,
    ["DEST"],
    "DEST",
)

df_train_hist = (
    df_train_hist
    .join(dest_history_features, on=["DEST", "FL_DATE"], how="left")
    .join(date_specific_priors, on="FL_DATE", how="left")
    .withColumn(
        "DEST_HIST_DELAY_RATE",
        F.coalesce(
            F.col("DEST_HIST_DELAY_RATE"),
            F.col("DATE_PRIOR_DELAY_RATE"),
            F.lit(INITIAL_DELAY_PRIOR),
        ),
    )
    .drop("DATE_PRIOR_DELAY_RATE")
)


#### Historical Route Delay Rate

`ROUTE_HIST_DELAY_RATE` uses only earlier dates on the same origin–destination route. Sparse routes are smoothed toward the causal prior available before that flight date.


In [0]:
route_history_features = causal_entity_history(
    df_train,
    ["ORIGIN", "DEST"],
    "ROUTE",
)

df_train_hist = (
    df_train_hist
    .join(
        route_history_features,
        on=["ORIGIN", "DEST", "FL_DATE"],
        how="left",
    )
    .join(date_specific_priors, on="FL_DATE", how="left")
    .withColumn(
        "ROUTE_HIST_DELAY_RATE",
        F.coalesce(
            F.col("ROUTE_HIST_DELAY_RATE"),
            F.col("DATE_PRIOR_DELAY_RATE"),
            F.lit(INITIAL_DELAY_PRIOR),
        ),
    )
    .drop("DATE_PRIOR_DELAY_RATE")
)

print("All training historical rates use strictly earlier dates.")


#### Apply Development-Period History to Later Validation and Test Data

September–October validation and November–December test records occur after the January–August development period. Their airline, airport, and route mappings may therefore use the complete development history without temporal leakage. Unseen categories fall back to the January–August development delay rate.


In [0]:
# Safe fallback for periods after the January–August development window.
global_training_delay_rate = (
    df_train
    .select(F.avg(F.col(TARGET_COLUMN).cast('double')).alias('RATE'))
    .first()['RATE']
)

# -------------------------------------------------------
# Create historical mappings from training data only
# -------------------------------------------------------

airline_training_map = (
    df_train
    .groupBy("OP_UNIQUE_CARRIER")
    .agg(
        F.count("*").alias("AIRLINE_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "AIRLINE_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "AIRLINE_HIST_DELAY_RATE",
        (
            F.col("AIRLINE_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("AIRLINE_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "OP_UNIQUE_CARRIER",
        "AIRLINE_HIST_DELAY_RATE",
    )
)

origin_training_map = (
    df_train
    .groupBy("ORIGIN")
    .agg(
        F.count("*").alias("ORIGIN_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "ORIGIN_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "ORIGIN_HIST_DELAY_RATE",
        (
            F.col("ORIGIN_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("ORIGIN_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "ORIGIN",
        "ORIGIN_HIST_DELAY_RATE",
    )
)

dest_training_map = (
    df_train
    .groupBy("DEST")
    .agg(
        F.count("*").alias("DEST_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "DEST_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "DEST_HIST_DELAY_RATE",
        (
            F.col("DEST_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("DEST_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "DEST",
        "DEST_HIST_DELAY_RATE",
    )
)

route_training_map = (
    df_train
    .groupBy("ORIGIN", "DEST")
    .agg(
        F.count("*").alias("ROUTE_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "ROUTE_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "ROUTE_HIST_DELAY_RATE",
        (
            F.col("ROUTE_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("ROUTE_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "ORIGIN",
        "DEST",
        "ROUTE_HIST_DELAY_RATE",
    )
)


def attach_training_history(dataset: DataFrame) -> DataFrame:
    """Attach historical rates calculated exclusively from training data."""
    return (
        dataset
        .join(
            airline_training_map,
            on="OP_UNIQUE_CARRIER",
            how="left",
        )
        .join(
            origin_training_map,
            on="ORIGIN",
            how="left",
        )
        .join(
            dest_training_map,
            on="DEST",
            how="left",
        )
        .join(
            route_training_map,
            on=["ORIGIN", "DEST"],
            how="left",
        )
        .fillna(
            {
                "AIRLINE_HIST_DELAY_RATE": global_training_delay_rate,
                "ORIGIN_HIST_DELAY_RATE": global_training_delay_rate,
                "DEST_HIST_DELAY_RATE": global_training_delay_rate,
                "ROUTE_HIST_DELAY_RATE": global_training_delay_rate,
            }
        )
    )


df_validation_hist = attach_training_history(df_validation)
df_test_hist = attach_training_history(df_test)

print(
    f"Validation rows after historical joins: "
    f"{df_validation_hist.count():,}"
)
print(
    f"Test rows after historical joins: "
    f"{df_test_hist.count():,}"
)

display(
    df_validation_hist
    .select(
        "FL_DATE",
        "OP_UNIQUE_CARRIER",
        "ORIGIN",
        "DEST",
        "AIRLINE_HIST_DELAY_RATE",
        "ORIGIN_HIST_DELAY_RATE",
        "DEST_HIST_DELAY_RATE",
        "ROUTE_HIST_DELAY_RATE",
        "ARR_DEL15",
    )
    .limit(20)
)


#### Validate historical performance features

The historical feature datasets are validated before categorical encoding and model training.

This check confirms that:

- Record counts remain unchanged after historical-feature joins
- No historical delay-rate features contain missing values
- All historical rates fall within the valid probability range of 0 to 1
- Training, validation, and test datasets contain the same historical feature columns


In [0]:
HISTORICAL_RATE_COLUMNS = list(cfg.MODEL_HISTORICAL_RATE_COLUMNS)

historical_validation_rows = []

for dataset_name, dataset in [
    ("TRAIN", df_train_hist),
    ("VALIDATION", df_validation_hist),
    ("TEST", df_test_hist),
]:
    summary_row = (
        dataset
        .select(
            F.lit(dataset_name).alias("DATASET"),
            F.count("*").alias("TOTAL_RECORDS"),
            *[
                F.sum(
                    F.when(F.col(column_name).isNull(), 1).otherwise(0)
                ).alias(f"{column_name}_NULLS")
                for column_name in HISTORICAL_RATE_COLUMNS
            ],
            *[
                F.sum(
                    F.when(
                        (F.col(column_name) < 0)
                        | (F.col(column_name) > 1),
                        1,
                    ).otherwise(0)
                ).alias(f"{column_name}_OUT_OF_RANGE")
                for column_name in HISTORICAL_RATE_COLUMNS
            ],
        )
    )

    historical_validation_rows.append(summary_row)

historical_validation_summary = historical_validation_rows[0]

for summary_row in historical_validation_rows[1:]:
    historical_validation_summary = (
        historical_validation_summary.unionByName(summary_row)
    )

display(historical_validation_summary)


#### Prepare Raw Predictor Columns for Standard Python

The leakage-safe historical training, validation, and test frames retain their original categorical and numerical predictor columns. Spark is used only to prepare, filter, and sample these large datasets.

Categorical encoding is deliberately deferred until after each chronological training fold has been sampled. A scikit-learn `ColumnTransformer` is fitted only on that fold's training observations and then applied to its later validation month. This keeps category learning inside the training boundary and avoids preprocessing leakage.


In [0]:
import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

from utils.model_training import prepare_hist_modeling_frame

CATEGORICAL_COLUMNS = list(cfg.MODEL_CATEGORICAL_COLUMNS)
NUMERICAL_COLUMNS = list(cfg.MODEL_NUMERICAL_COLUMNS)
MODEL_INPUT_COLUMNS = list(cfg.MODEL_INPUT_COLUMNS)

df_train_hist = prepare_hist_modeling_frame(df_train_hist)
df_validation_hist = prepare_hist_modeling_frame(df_validation_hist)
df_test_hist = prepare_hist_modeling_frame(df_test_hist)

df_train_prepared = df_train_hist.select(
    "FL_DATE", *MODEL_INPUT_COLUMNS, TARGET_COLUMN
)
df_validation_prepared = df_validation_hist.select(
    "FL_DATE", *MODEL_INPUT_COLUMNS, TARGET_COLUMN
)
df_test_prepared = df_test_hist.select(
    "FL_DATE", *MODEL_INPUT_COLUMNS, TARGET_COLUMN
)

print("Raw predictor datasets prepared for scikit-learn preprocessing.")
print(f"Categorical predictors: {len(CATEGORICAL_COLUMNS)}")
print(f"Numerical predictors: {len(NUMERICAL_COLUMNS)}")
print(f"Total predictors: {len(MODEL_INPUT_COLUMNS)}")


#### Validate the Raw Model Datasets

The raw predictor schema is validated before sampling. Each dataset must contain the flight date, target, and exactly the predictor columns expected by the standard-Python preprocessing pipeline.

No categorical mappings or numerical imputation values are learned at this stage.


In [0]:
required_model_columns = {
    "FL_DATE", TARGET_COLUMN, *MODEL_INPUT_COLUMNS
}

for dataframe_name, dataframe in {
    "df_train_prepared": df_train_prepared,
    "df_validation_prepared": df_validation_prepared,
    "df_test_prepared": df_test_prepared,
}.items():
    missing_columns = sorted(
        required_model_columns - set(dataframe.columns)
    )
    if missing_columns:
        raise ValueError(
            f"{dataframe_name} is missing columns: {missing_columns}"
        )
    if dataframe.limit(1).count() == 0:
        raise ValueError(f"{dataframe_name} contains no rows.")
    print(f"{dataframe_name} schema validated successfully.")


#### Standard-Python Preprocessing Boundary

After bounded sampling, raw Spark rows are collected into pandas. Scikit-learn then performs median numerical imputation, most-frequent categorical imputation, sparse one-hot encoding with unknown-category handling, and sparse-safe numerical scaling.

Every preprocessing transformer is fitted on training rows only. Logistic Regression, Random Forest, and XGBoost receive the same transformed sparse matrices within each fold.


In [0]:
print("Standard-Python preprocessing inputs are ready.")
print(f"Training rows: {df_train_prepared.count():,}")
print(f"Validation rows: {df_validation_prepared.count():,}")
print(f"Test rows: {df_test_prepared.count():,}")

display(
    df_train_prepared.select(
        "FL_DATE",
        *MODEL_INPUT_COLUMNS,
        TARGET_COLUMN,
    ).limit(10)
)


## Standard-Python Candidate Modeling

Logistic Regression, Random Forest, and XGBoost are implemented using standard Python libraries rather than Spark ML. Spark remains responsible only for loading the large Delta tables, applying the leakage-safe feature engineering workflow, and creating bounded samples that can be safely collected by the Databricks Free Edition driver.

The three algorithms receive the same SciPy sparse feature matrices, chronological folds, validation observations, evaluation metrics, and selection policy. This shared design removes implementation differences that could otherwise make the comparison difficult to interpret.

The candidate estimators are:

- `sklearn.linear_model.LogisticRegression`
- `sklearn.ensemble.RandomForestClassifier`
- `xgboost.XGBClassifier`


### Install Standard-Python Modeling Dependencies

Databricks Free Edition may not include XGBoost in a new Python session. The following installation cell runs before any modeling-library imports so that **Run All** can prepare the required standard-Python environment automatically.

The XGBoost version is pinned for reproducibility. Databricks may display routine package-installation messages while this cell runs.


In [0]:
%pip install --quiet xgboost==2.1.4 scipy scikit-learn


In [0]:
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import random
    import warnings

    from scipy.sparse import csr_matrix
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.calibration import calibration_curve
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        accuracy_score,
        average_precision_score,
        brier_score_loss,
        confusion_matrix,
        precision_recall_fscore_support,
        roc_auc_score,
    )
    from sklearn.model_selection import ParameterGrid
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from xgboost import XGBClassifier
except ImportError as error:
    raise ImportError(
        "Notebook 07 requires NumPy, SciPy, scikit-learn, and XGBoost. "
        "Run `%pip install xgboost==2.1.4 scipy scikit-learn`, restart "
        "Python once, and rerun the notebook from the beginning."
    ) from error

import ast
import time

from pyspark.sql import functions as F

RANDOM_SEED = cfg.RANDOM_SEED
LOCAL_TRAIN_MAX_ROWS = 200_000
LOCAL_VALIDATION_MAX_ROWS = 50_000

print("Standard-Python modeling libraries loaded successfully.")
print(f"Maximum local training rows per fold: {LOCAL_TRAIN_MAX_ROWS:,}")
print(
    "Maximum local validation rows per fold: "
    f"{LOCAL_VALIDATION_MAX_ROWS:,}"
)

## Reproducible Standard-Python Preprocessing Utilities

Bounded raw Spark samples are collected into pandas and transformed by scikit-learn. Categorical variables are one-hot encoded into a sparse matrix; numerical variables are imputed and scaled without centering so the combined representation remains sparse.

Sampling is uniform and reproducible. It does not rebalance the validation data. Consequently, the validation samples retain the natural proportion of delayed and on-time flights.


In [0]:
def bounded_uniform_sample(
    dataframe,
    maximum_rows,
    *,
    seed,
):
    """Return a reproducible uniform sample with at most maximum_rows."""
    row_count = dataframe.count()

    if row_count == 0:
        raise ValueError("Cannot sample an empty Spark DataFrame.")

    if row_count <= maximum_rows:
        return dataframe

    sampling_fraction = min(
        1.0,
        (maximum_rows * 1.10) / row_count,
    )

    return (
        dataframe
        .sample(
            withReplacement=False,
            fraction=sampling_fraction,
            seed=seed,
        )
        .limit(maximum_rows)
    )


def spark_sample_to_pandas(dataframe):
    """Collect one bounded raw Spark sample into pandas."""
    local_frame = dataframe.select(
        *MODEL_INPUT_COLUMNS,
        TARGET_COLUMN,
    ).toPandas()
    if local_frame.empty:
        raise ValueError("The local modeling sample contains zero rows.")

    for column_name in CATEGORICAL_COLUMNS:
        categorical_values = local_frame[column_name].astype(
            "object"
        )
        local_frame[column_name] = categorical_values.where(
            pd.notna(categorical_values),
            np.nan,
        )
    for column_name in NUMERICAL_COLUMNS:
        local_frame[column_name] = pd.to_numeric(
            local_frame[column_name], errors="coerce"
        )
    return local_frame


def build_sklearn_preprocessor():
    """Build a fresh transformer fitted only on training rows."""
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )
    numerical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler(with_mean=False)),
        ]
    )
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                categorical_pipeline,
                CATEGORICAL_COLUMNS,
            ),
            (
                "numerical",
                numerical_pipeline,
                NUMERICAL_COLUMNS,
            ),
        ],
        sparse_threshold=1.0,
    )


def prepare_training_validation_matrices(
    training_dataframe,
    validation_dataframe,
):
    """Fit preprocessing on training and transform later validation."""
    training_local = spark_sample_to_pandas(training_dataframe)
    validation_local = spark_sample_to_pandas(validation_dataframe)

    preprocessor = build_sklearn_preprocessor()
    X_training = csr_matrix(
        preprocessor.fit_transform(
            training_local[MODEL_INPUT_COLUMNS]
        ),
        dtype=np.float32,
    )
    X_validation = csr_matrix(
        preprocessor.transform(
            validation_local[MODEL_INPUT_COLUMNS]
        ),
        dtype=np.float32,
    )

    y_training = training_local[TARGET_COLUMN].to_numpy(dtype=np.int8)
    y_validation = validation_local[TARGET_COLUMN].to_numpy(dtype=np.int8)

    return (
        preprocessor,
        X_training,
        y_training,
        X_validation,
        y_validation,
    )


def evaluate_python_classifier(model, X_validation, y_validation):
    """Evaluate a fitted binary classifier using the shared metrics."""
    predictions = model.predict(X_validation).astype(np.int8)
    probabilities = model.predict_proba(X_validation)[:, 1]

    top_count = max(1, int(np.ceil(len(y_validation) * 0.10)))
    top_indices = np.argsort(probabilities)[-top_count:]
    total_delays = int(np.sum(y_validation == 1))
    top_delays = int(np.sum(y_validation[top_indices] == 1))
    overall_delay_rate = float(np.mean(y_validation))
    top_delay_rate = float(np.mean(y_validation[top_indices]))

    weighted_precision, weighted_recall, weighted_f1, _ = (
        precision_recall_fscore_support(
            y_validation,
            predictions,
            average="weighted",
            zero_division=0,
        )
    )

    delay_precision, delay_recall, delay_f1, _ = (
        precision_recall_fscore_support(
            y_validation,
            predictions,
            average="binary",
            pos_label=1,
            zero_division=0,
        )
    )

    return {
        "ACCURACY": float(
            accuracy_score(y_validation, predictions)
        ),
        "PRECISION": float(weighted_precision),
        "RECALL": float(weighted_recall),
        "F1_SCORE": float(weighted_f1),
        "ROC_AUC": float(
            roc_auc_score(y_validation, probabilities)
        ),
        "PR_AUC": float(
            average_precision_score(
                y_validation,
                probabilities,
            )
        ),
        "BRIER_SCORE": float(
            brier_score_loss(y_validation, probabilities)
        ),
        "TOP_10_RECALL": float(
            top_delays / total_delays if total_delays else 0.0
        ),
        "TOP_10_LIFT": float(
            top_delay_rate / overall_delay_rate
            if overall_delay_rate else 0.0
        ),
        "DELAY_PRECISION": float(delay_precision),
        "DELAY_RECALL": float(delay_recall),
        "DELAY_F1": float(delay_f1),
    }


def positive_class_weight(labels):
    """Return the negative-to-positive ratio for XGBoost."""
    positive_count = int(np.sum(labels == 1))
    negative_count = int(np.sum(labels == 0))

    if positive_count == 0 or negative_count == 0:
        raise ValueError(
            "Both target classes must be present in every training fold."
        )

    return negative_count / positive_count


def confusion_matrix_table(y_true, y_predicted):
    """Return a clearly labeled binary confusion matrix."""
    matrix = confusion_matrix(
        y_true,
        y_predicted,
        labels=[0, 1],
    )
    return pd.DataFrame(
        matrix,
        index=["Actual On Time (0)", "Actual Delayed (1)"],
        columns=["Predicted On Time (0)", "Predicted Delayed (1)"],
    )


def confusion_count_table(y_true, y_predicted):
    """Return explicit TN, FP, FN, and TP counts with definitions."""
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_predicted,
        labels=[0, 1],
    ).ravel()

    return pd.DataFrame(
        [
            {
                "CONFUSION_TERM": "TN",
                "MEANING": "Actual on-time flight predicted as on time",
                "COUNT": int(tn),
            },
            {
                "CONFUSION_TERM": "FP",
                "MEANING": "Actual on-time flight predicted as delayed",
                "COUNT": int(fp),
            },
            {
                "CONFUSION_TERM": "FN",
                "MEANING": "Actual delayed flight predicted as on time",
                "COUNT": int(fn),
            },
            {
                "CONFUSION_TERM": "TP",
                "MEANING": "Actual delayed flight predicted as delayed",
                "COUNT": int(tp),
            },
        ]
    )


print("Shared local-matrix and evaluation utilities created.")


## Class-Imbalance Strategy

Delayed flights form the minority class. A classifier can therefore achieve high overall Accuracy by predicting most flights as on time while missing many actual delays.

Class imbalance is handled only within each training dataset:

- Logistic Regression uses `class_weight="balanced"`.
- Random Forest uses `class_weight="balanced_subsample"`.
- XGBoost uses `scale_pos_weight`, calculated as the number of on-time training observations divided by the number of delayed training observations.

Validation observations are not oversampled, undersampled, or synthetically generated. Keeping the validation distribution unchanged produces metrics that reflect realistic flight operations and prevents information from the validation period from influencing model training.

Delayed-flight Recall, delayed-flight F1-score, and PR AUC are reported in addition to overall and weighted metrics. These metrics expose minority-class performance that Accuracy alone may conceal.


## Metric Definitions and Delayed-Class Formulas

The target is binary: `0` represents an on-time flight and `1` represents a delayed flight. Metrics beginning with `DELAY_` evaluate class `1` specifically, whereas the unprefixed Precision, Recall, and F1 columns are weighted averages across both classes.

Let `TP` be delayed flights correctly predicted as delayed, `FN` be delayed flights incorrectly predicted as on time, and `FP` be on-time flights incorrectly predicted as delayed.

**Delayed-flight Recall**

`DELAY_RECALL = TP / (TP + FN)`

This answers: Of all flights that actually became delayed, what proportion did the model detect?

**Delayed-flight Precision**

`DELAY_PRECISION = TP / (TP + FP)`

This answers: Of all flights flagged as delayed, what proportion actually became delayed?

**Delayed-flight F1-score**

`DELAY_F1 = 2 * (DELAY_PRECISION * DELAY_RECALL) / (DELAY_PRECISION + DELAY_RECALL)`

This balances detecting delayed flights against avoiding excessive false delay alerts.

The result-table names have the following meanings:

- `ACCURACY`: proportion of all predictions that are correct.
- `PRECISION`: class-frequency-weighted Precision across classes `0` and `1`.
- `RECALL`: class-frequency-weighted Recall across classes `0` and `1`; in single-label classification it equals Accuracy and is not the delayed-class Recall.
- `F1_SCORE`: class-frequency-weighted F1 across classes `0` and `1`.
- `DELAY_PRECISION`, `DELAY_RECALL`, and `DELAY_F1`: metrics for delayed flights only.
- `ROC_AUC`: discrimination between on-time and delayed flights across thresholds.
- `PR_AUC`: probability-ranking quality for the minority delayed-flight class.

Delayed-flight Recall is emphasized because false negatives are actual delays that receive no warning. It is interpreted together with delayed-flight F1 and PR AUC so that a model is not rewarded merely for flagging nearly every flight as delayed.


## Shared Baseline Dataset

Before hyperparameter tuning, the three standard-Python algorithms are trained and evaluated using the same bounded chronological datasets. Training observations come only from the January–August development period. Validation observations come from the later September–October validation period.

The same feature matrix and labels are reused across all three algorithms, ensuring that baseline differences arise from the algorithms rather than from different samples.


In [0]:
baseline_training_df = bounded_uniform_sample(
    df_train_prepared.select(*MODEL_INPUT_COLUMNS, TARGET_COLUMN),
    LOCAL_TRAIN_MAX_ROWS,
    seed=RANDOM_SEED,
)
baseline_validation_df = bounded_uniform_sample(
    df_validation_prepared.select(*MODEL_INPUT_COLUMNS, TARGET_COLUMN),
    LOCAL_VALIDATION_MAX_ROWS,
    seed=RANDOM_SEED + 1,
)

(
    baseline_preprocessor,
    X_baseline_train,
    y_baseline_train,
    X_baseline_validation,
    y_baseline_validation,
) = prepare_training_validation_matrices(
    baseline_training_df,
    baseline_validation_df,
)

print(f"Shared baseline training rows: {X_baseline_train.shape[0]:,}")
print(
    "Shared baseline validation rows: "
    f"{X_baseline_validation.shape[0]:,}"
)
print(
    "Baseline training delayed-flight rate: "
    f"{np.mean(y_baseline_train):.4f}"
)
print(
    "Baseline validation delayed-flight rate: "
    f"{np.mean(y_baseline_validation):.4f}"
)


## Train and Evaluate Standard-Python Baselines

The baseline configurations provide an initial reference before tuning. Class-imbalance controls are enabled for every trainable candidate, while the majority-class baseline demonstrates why Accuracy alone is insufficient.


In [0]:
majority_class = int(
    np.bincount(y_baseline_train).argmax()
)
majority_predictions = np.full(
    y_baseline_validation.shape,
    majority_class,
    dtype=np.int8,
)

majority_weighted_precision, majority_weighted_recall, majority_weighted_f1, _ = (
    precision_recall_fscore_support(
        y_baseline_validation,
        majority_predictions,
        average="weighted",
        zero_division=0,
    )
)

baseline_rows = [
    {
        "MODEL": "Majority Class Baseline",
        "ACCURACY": float(
            accuracy_score(
                y_baseline_validation,
                majority_predictions,
            )
        ),
        "PRECISION": float(majority_weighted_precision),
        "RECALL": float(majority_weighted_recall),
        "F1_SCORE": float(majority_weighted_f1),
        "ROC_AUC": None,
        "PR_AUC": None,
        "BRIER_SCORE": float(
            brier_score_loss(
                y_baseline_validation,
                majority_predictions.astype(float),
            )
        ),
        "TOP_10_RECALL": 0.0,
        "TOP_10_LIFT": 0.0,
        "DELAY_PRECISION": 0.0,
        "DELAY_RECALL": 0.0,
        "DELAY_F1": 0.0,
    }
]

baseline_estimators = {
    "Logistic Regression (Baseline)": LogisticRegression(
        C=1.0,
        penalty="l2",
        solver="liblinear",
        class_weight="balanced",
        max_iter=500,
        random_state=RANDOM_SEED,
    ),
    "Random Forest (Baseline)": RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
    "XGBoost (Baseline)": XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.10,
        min_child_weight=1,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        scale_pos_weight=positive_class_weight(y_baseline_train),
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
}

baseline_predictions = {}

for model_name, estimator in baseline_estimators.items():
    estimator.fit(X_baseline_train, y_baseline_train)
    baseline_predictions[model_name] = (
        estimator.predict(X_baseline_validation).astype(np.int8)
    )
    metrics = evaluate_python_classifier(
        estimator,
        X_baseline_validation,
        y_baseline_validation,
    )
    baseline_rows.append(
        {
            "MODEL": model_name,
            **metrics,
        }
    )

model_comparison = spark.createDataFrame(baseline_rows)
display(model_comparison.orderBy("MODEL"))


## Baseline Confusion Matrices

The confusion matrices below show how each untuned algorithm classified the same chronological validation observations. Rows represent actual outcomes and columns represent predicted outcomes.

- The upper-left cell is the number of correctly identified on-time flights (true negatives).
- The upper-right cell is the number of on-time flights incorrectly flagged as delayed (false positives).
- The lower-left cell is the number of delayed flights missed by the model (false negatives).
- The lower-right cell is the number of delayed flights correctly identified (true positives).

For this project, the lower-left cell is especially important because it contains actual delays that would receive no operational warning.


In [0]:
for model_name, predictions in baseline_predictions.items():
    print(f"Baseline confusion matrix: {model_name}")
    display(
        confusion_matrix_table(
            y_baseline_validation,
            predictions,
        )
    )
    print(f"Baseline TN/FP/FN/TP counts: {model_name}")
    display(
        confusion_count_table(
            y_baseline_validation,
            predictions,
        )
    )


## Baseline Comparison Interpretation

The majority-class baseline is included as a diagnostic reference rather than a viable operational model. Its Accuracy can appear strong because most flights are on time, but its delayed-flight Recall is zero.

The three machine-learning baselines use identical local training and validation matrices and explicitly account for class imbalance. Their relative performance therefore provides a fair preliminary comparison. Final selection is not based on this single validation period; it is based on average performance across the shared chronological cross-validation folds below.


## Unified Chronological Cross-Validation

One cross-validation workflow is used for Logistic Regression, Random Forest, and XGBoost.

The January–August development period is divided into five expanding-window folds:

| Fold | Training period | Validation period |
|---|---|---|
| 1 | January–March 2025 | April 2025 |
| 2 | January–April 2025 | May 2025 |
| 3 | January–May 2025 | June 2025 |
| 4 | January–June 2025 | July 2025 |
| 5 | January–July 2025 | August 2025 |

For each fold, the chronological date boundaries are applied before sampling. One reproducible uniform training sample and one reproducible uniform validation sample are then created and converted to local sparse matrices. These exact matrices are reused by every algorithm and every hyperparameter configuration.

This arrangement prevents future observations from entering earlier training periods, keeps validation distributions natural, and avoids algorithm-specific sampling differences.


In [0]:
TUNING_FOLDS = [
    {
        "train_end": "2025-03-31",
        "validation_start": "2025-04-01",
        "validation_end": "2025-04-30",
    },
    {
        "train_end": "2025-04-30",
        "validation_start": "2025-05-01",
        "validation_end": "2025-05-31",
    },
    {
        "train_end": "2025-05-31",
        "validation_start": "2025-06-01",
        "validation_end": "2025-06-30",
    },
    {
        "train_end": "2025-06-30",
        "validation_start": "2025-07-01",
        "validation_end": "2025-07-31",
    },
    {
        "train_end": "2025-07-31",
        "validation_start": "2025-08-01",
        "validation_end": "2025-08-31",
    },
]

df_tuning_complete = (
    df_train_prepared
    .select("FL_DATE", *MODEL_INPUT_COLUMNS, TARGET_COLUMN)
)

LOCAL_CV_FOLDS = []

for fold_number, fold in enumerate(TUNING_FOLDS, start=1):
    complete_training_fold = df_tuning_complete.filter(
        F.col("FL_DATE")
        <= F.to_date(F.lit(fold["train_end"]))
    )
    complete_validation_fold = df_tuning_complete.filter(
        F.col("FL_DATE").between(
            F.to_date(F.lit(fold["validation_start"])),
            F.to_date(F.lit(fold["validation_end"])),
        )
    )

    sampled_training_fold = bounded_uniform_sample(
        complete_training_fold,
        LOCAL_TRAIN_MAX_ROWS,
        seed=RANDOM_SEED + fold_number * 10,
    )
    sampled_validation_fold = bounded_uniform_sample(
        complete_validation_fold,
        LOCAL_VALIDATION_MAX_ROWS,
        seed=RANDOM_SEED + fold_number * 10 + 1,
    )

    (
        fold_preprocessor,
        X_train_fold,
        y_train_fold,
        X_validation_fold,
        y_validation_fold,
    ) = prepare_training_validation_matrices(
        sampled_training_fold,
        sampled_validation_fold,
    )

    LOCAL_CV_FOLDS.append(
        {
            "fold": fold_number,
            "X_train": X_train_fold,
            "y_train": y_train_fold,
            "X_validation": X_validation_fold,
            "y_validation": y_validation_fold,
            "preprocessor": fold_preprocessor,
            "scale_pos_weight": positive_class_weight(
                y_train_fold
            ),
        }
    )

    print(
        f"Fold {fold_number}: "
        f"{X_train_fold.shape[0]:,} training rows, "
        f"{X_validation_fold.shape[0]:,} validation rows, "
        f"training delay rate={np.mean(y_train_fold):.4f}, "
        f"validation delay rate={np.mean(y_validation_fold):.4f}"
    )

print("Shared chronological folds prepared for all algorithms.")


## Runtime-Aware Deep, Two-Stage Search

The search is deliberately deeper without evaluating every candidate on every large fold. Each algorithm receives a randomized broad pool and a unique local refinement pool:

- Logistic Regression: 24 broad candidates and 10 refinement candidates.
- Random Forest: 36 broad candidates and 12 refinement candidates.
- XGBoost: 54 broad candidates and 18 refinement candidates.

All candidates are first screened on the first and fifth chronological folds using smaller, identical data slices. The strongest broad candidates and refinement candidates are then promoted to the complete five-fold evaluation. This staged design preserves meaningful search breadth while controlling Databricks Free Edition runtime.

For every fully evaluated configuration, the operating threshold is chosen from pooled out-of-fold predictions using the same rule: maximize delayed-class F2 while satisfying a minimum delayed-flight Recall of 0.60 whenever feasible. This avoids unfairly comparing models only at the default threshold of 0.50.

In [0]:
SEARCH_RANDOM_SEED = RANDOM_SEED + 700
SCREEN_FOLD_INDICES = [0, 4]
SCREEN_TRAIN_ROWS = 60_000
SCREEN_VALIDATION_ROWS = 30_000
MIN_OPERATING_RECALL = 0.60
PR_AUC_TOLERANCE = 0.02
MAX_PR_AUC_STD = 0.09
MIN_FINAL_DELAY_RECALL = 0.50


def sampled_parameter_pool(space, count, seed):
    candidates = list(ParameterGrid(space))
    rng = random.Random(seed)
    rng.shuffle(candidates)
    return candidates[: min(count, len(candidates))]


LOGISTIC_REGRESSION_BROAD_GRID = sampled_parameter_pool(
    [
        {
            "C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0],
            "penalty": ["l2"],
            "solver": ["lbfgs"],
            "imbalance_strategy": ["natural", "class_weight", "undersample"],
        },
        {
            "C": [0.003, 0.01, 0.03, 0.1, 0.3, 1.0],
            "penalty": ["l1", "l2"],
            "solver": ["liblinear"],
            "imbalance_strategy": ["natural", "class_weight", "undersample"],
        },
    ],
    24,
    SEARCH_RANDOM_SEED,
)

RANDOM_FOREST_BROAD_GRID = sampled_parameter_pool(
    {
        "n_estimators": [150, 250, 350, 500],
        "max_depth": [8, 12, 16, 24, None],
        "min_samples_leaf": [1, 2, 5, 10],
        "max_features": ["sqrt", "log2", 0.5],
        "criterion": ["gini", "entropy"],
        "imbalance_strategy": ["natural", "class_weight", "undersample"],
    },
    36,
    SEARCH_RANDOM_SEED + 1,
)

XGBOOST_BROAD_GRID = sampled_parameter_pool(
    {
        "n_estimators": [150, 250, 350, 500, 700],
        "max_depth": [3, 5, 7, 9],
        "learning_rate": [0.02, 0.04, 0.07, 0.10],
        "min_child_weight": [1, 3, 5, 8],
        "subsample": [0.7, 0.85, 1.0],
        "colsample_bytree": [0.7, 0.85, 1.0],
        "reg_lambda": [1.0, 3.0, 7.0],
        "gamma": [0.0, 0.2, 0.5],
        "imbalance_strategy": ["natural", "class_weight", "undersample"],
    },
    54,
    SEARCH_RANDOM_SEED + 2,
)

print("Deep staged-search pools created:")
print(f"  Logistic Regression broad candidates: {len(LOGISTIC_REGRESSION_BROAD_GRID)}")
print(f"  Random Forest broad candidates: {len(RANDOM_FOREST_BROAD_GRID)}")
print(f"  XGBoost broad candidates: {len(XGBOOST_BROAD_GRID)}")

## Shared Estimator Builders and Imbalance Treatments

Every algorithm is evaluated under the same three class-imbalance alternatives: the natural distribution, class weighting, and training-only undersampling. Validation observations always retain their natural distribution. Logistic Regression uses a larger iteration limit, and configurations that still issue a convergence warning are excluded from final eligibility.

In [0]:
def apply_imbalance_strategy(X, y, strategy, seed):
    if strategy != "undersample":
        return X, y
    positive = np.flatnonzero(y == 1)
    negative = np.flatnonzero(y == 0)
    if len(positive) == 0 or len(negative) == 0:
        return X, y
    rng = np.random.default_rng(seed)
    retained_negative = rng.choice(
        negative, size=min(len(negative), len(positive)), replace=False
    )
    retained = np.concatenate([positive, retained_negative])
    rng.shuffle(retained)
    return X[retained], y[retained]


def build_logistic_regression(params, y_train=None):
    p = dict(params)
    strategy = p.pop("imbalance_strategy")
    return LogisticRegression(
        **p,
        class_weight="balanced" if strategy == "class_weight" else None,
        max_iter=5_000,
        random_state=RANDOM_SEED,
    )


def build_random_forest(params, y_train=None):
    p = dict(params)
    strategy = p.pop("imbalance_strategy")
    return RandomForestClassifier(
        **p,
        class_weight="balanced" if strategy == "class_weight" else None,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )


def build_xgboost(params, y_train=None):
    p = dict(params)
    strategy = p.pop("imbalance_strategy")
    scale = positive_class_weight(y_train) if strategy == "class_weight" else 1.0
    return XGBClassifier(
        **p,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        scale_pos_weight=scale,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )


ESTIMATOR_BUILDERS = {
    "Logistic Regression": build_logistic_regression,
    "Random Forest": build_random_forest,
    "XGBoost": build_xgboost,
}

## Shared Staged-Tuning and Threshold-Selection Framework

The framework screens broad candidates, evaluates promoted candidates on all five folds, generates unique refinements around the strongest broad configuration, and evaluates the promoted refinements on all five folds. Mean and standard deviation are retained for temporal-stability assessment.

The delayed-class threshold is selected identically for every candidate from pooled out-of-fold probabilities. The threshold maximizes F2, which gives Recall more weight than Precision, subject to the common minimum-Recall requirement. PR-AUC and ROC-AUC remain threshold independent.

In [0]:
RESULT_METRICS = [
    "ACCURACY", "PRECISION", "RECALL", "F1_SCORE", "ROC_AUC", "PR_AUC",
    "BRIER_SCORE", "DELAY_PRECISION", "DELAY_RECALL", "DELAY_F1",
    "TOP_10_RECALL", "TOP_10_LIFT", "TRAINING_SECONDS",
]


def metrics_from_probabilities(labels, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(np.int8)
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0
    )
    delay_precision, delay_recall, delay_f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary", pos_label=1, zero_division=0
    )
    top_count = max(1, int(np.ceil(0.10 * len(labels))))
    top_indices = np.argpartition(probabilities, -top_count)[-top_count:]
    positives = max(1, int(np.sum(labels == 1)))
    top_recall = float(np.sum(labels[top_indices] == 1) / positives)
    prevalence = max(float(np.mean(labels)), np.finfo(float).eps)
    lift = float(np.mean(labels[top_indices]) / prevalence)
    return {
        "ACCURACY": float(accuracy_score(labels, predictions)),
        "PRECISION": float(weighted_precision), "RECALL": float(weighted_recall),
        "F1_SCORE": float(weighted_f1),
        "ROC_AUC": float(roc_auc_score(labels, probabilities)),
        "PR_AUC": float(average_precision_score(labels, probabilities)),
        "BRIER_SCORE": float(brier_score_loss(labels, probabilities)),
        "DELAY_PRECISION": float(delay_precision), "DELAY_RECALL": float(delay_recall),
        "DELAY_F1": float(delay_f1), "TOP_10_RECALL": top_recall, "TOP_10_LIFT": lift,
    }


def choose_operating_threshold(labels, probabilities, minimum_recall=MIN_OPERATING_RECALL):
    candidate_thresholds = np.unique(
        np.concatenate(([0.05, 0.50, 0.95], np.quantile(probabilities, np.linspace(0.02, 0.98, 97))))
    )
    scored = []
    for threshold in candidate_thresholds:
        prediction = probabilities >= threshold
        precision, recall, _, _ = precision_recall_fscore_support(
            labels, prediction, average="binary", pos_label=1, zero_division=0
        )
        f2 = 5 * precision * recall / max(4 * precision + recall, np.finfo(float).eps)
        scored.append((float(threshold), float(precision), float(recall), float(f2)))
    feasible = [row for row in scored if row[2] >= minimum_recall]
    pool = feasible if feasible else scored
    return max(pool, key=lambda row: (row[3], row[1], row[0]))[0]


def evaluate_configuration(model_name, params, fold_indices, *, screening=False):
    labels_by_fold, probabilities_by_fold, seconds_by_fold = [], [], []
    converged = True
    for fold_index in fold_indices:
        fold = LOCAL_CV_FOLDS[fold_index]
        X_train, y_train = fold["X_train"], fold["y_train"]
        X_validation, y_validation = fold["X_validation"], fold["y_validation"]
        if screening:
            X_train, y_train = X_train[:SCREEN_TRAIN_ROWS], y_train[:SCREEN_TRAIN_ROWS]
            X_validation = X_validation[:SCREEN_VALIDATION_ROWS]
            y_validation = y_validation[:SCREEN_VALIDATION_ROWS]
        strategy = params["imbalance_strategy"]
        X_fit, y_fit = apply_imbalance_strategy(
            X_train, y_train, strategy, RANDOM_SEED + fold_index
        )
        estimator = ESTIMATOR_BUILDERS[model_name](params, y_fit)
        start = time.perf_counter()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            estimator.fit(X_fit, y_fit)
        converged = converged and not any(
            issubclass(item.category, ConvergenceWarning) for item in caught
        )
        seconds_by_fold.append(time.perf_counter() - start)
        labels_by_fold.append(np.asarray(y_validation, dtype=np.int8))
        probabilities_by_fold.append(estimator.predict_proba(X_validation)[:, 1])
    pooled_labels = np.concatenate(labels_by_fold)
    pooled_probabilities = np.concatenate(probabilities_by_fold)
    threshold = choose_operating_threshold(pooled_labels, pooled_probabilities)
    fold_metrics = [
        metrics_from_probabilities(y, p, threshold)
        for y, p in zip(labels_by_fold, probabilities_by_fold)
    ]
    row = {
        "MODEL": model_name, "PARAMETERS": str(params),
        "OPERATING_THRESHOLD": float(threshold), "CONVERGED": bool(converged),
        "FOLD_COUNT": int(len(fold_indices)),
    }
    for metric in RESULT_METRICS:
        values = seconds_by_fold if metric == "TRAINING_SECONDS" else [m[metric] for m in fold_metrics]
        row[metric] = float(np.mean(values))
        row[f"{metric}_STD"] = float(np.std(values, ddof=0))
    return row


def screening_key(row):
    return (-row["PR_AUC"], -row["DELAY_F1"], row["BRIER_SCORE"], row["TRAINING_SECONDS"])


def unique_parameters(rows):
    result, seen = [], set()
    for row in rows:
        key = tuple(sorted(row.items()))
        if key not in seen:
            seen.add(key)
            result.append(row)
    return result


def refinement_candidates(model_name, anchor, count, seed):
    strategy = anchor["imbalance_strategy"]
    if model_name == "Logistic Regression":
        c = float(anchor["C"])
        candidates = [
            {**anchor, "C": max(1e-5, c * factor), "imbalance_strategy": imbalance}
            for factor in [0.35, 0.5, 0.75, 1.25, 1.5, 2.0]
            for imbalance in [strategy, "natural", "class_weight", "undersample"]
        ]
    elif model_name == "Random Forest":
        depth = anchor["max_depth"] if anchor["max_depth"] is not None else 24
        candidates = [
            {**anchor, "n_estimators": trees, "max_depth": d, "min_samples_leaf": leaf}
            for trees in sorted(set([max(100, anchor["n_estimators"] - 100), anchor["n_estimators"] + 100, anchor["n_estimators"] + 250]))
            for d in sorted(set([max(4, depth - 4), depth, depth + 4]))
            for leaf in sorted(set([max(1, anchor["min_samples_leaf"] - 1), anchor["min_samples_leaf"], anchor["min_samples_leaf"] + 2]))
        ]
    else:
        candidates = [
            {**anchor, "n_estimators": trees, "max_depth": depth, "learning_rate": rate,
             "min_child_weight": child, "gamma": gamma}
            for trees in sorted(set([max(100, anchor["n_estimators"] - 100), anchor["n_estimators"] + 100, anchor["n_estimators"] + 250]))
            for depth in sorted(set([max(2, anchor["max_depth"] - 2), anchor["max_depth"], anchor["max_depth"] + 2]))
            for rate in sorted(set([max(0.01, anchor["learning_rate"] * 0.6), anchor["learning_rate"], min(0.2, anchor["learning_rate"] * 1.4)]))
            for child in sorted(set([max(1, anchor["min_child_weight"] - 2), anchor["min_child_weight"] + 2]))
            for gamma in sorted(set([max(0.0, anchor["gamma"] - 0.2), anchor["gamma"] + 0.2]))
        ]
    candidates = [p for p in unique_parameters(candidates) if p != anchor]
    rng = random.Random(seed)
    rng.shuffle(candidates)
    return candidates[:count]


def run_staged_search(model_name, broad_pool, broad_promotions, refinement_count, refinement_promotions):
    print(f"Screening {len(broad_pool)} broad {model_name} candidates on folds 1 and 5.")
    broad_screen = [
        {**evaluate_configuration(model_name, p, SCREEN_FOLD_INDICES, screening=True), "SEARCH_STAGE": "broad_screen"}
        for p in broad_pool
    ]
    promoted = sorted(broad_screen, key=screening_key)[:broad_promotions]
    full_broad = [
        {**evaluate_configuration(model_name, ast.literal_eval(r["PARAMETERS"]), range(5)), "SEARCH_STAGE": "broad"}
        for r in promoted
    ]
    anchor_row = min(full_broad, key=screening_key)
    anchor = ast.literal_eval(anchor_row["PARAMETERS"])
    refinements = refinement_candidates(
        model_name, anchor, refinement_count, SEARCH_RANDOM_SEED + len(model_name)
    )
    print(f"Screening {len(refinements)} unique {model_name} refinements.")
    refinement_screen = [
        {**evaluate_configuration(model_name, p, SCREEN_FOLD_INDICES, screening=True), "SEARCH_STAGE": "refinement_screen"}
        for p in refinements
    ]
    promoted_refinements = sorted(refinement_screen, key=screening_key)[:refinement_promotions]
    full_refinements = [
        {**evaluate_configuration(model_name, ast.literal_eval(r["PARAMETERS"]), range(5)), "SEARCH_STAGE": "refinement"}
        for r in promoted_refinements
    ]
    full_rows = full_broad + full_refinements
    dataframe = spark.createDataFrame(full_rows).orderBy(
        F.desc("PR_AUC"), F.desc("DELAY_F1"), F.asc("BRIER_SCORE")
    )
    return dataframe, full_rows, spark.createDataFrame(broad_screen + refinement_screen)

## Tune Logistic Regression

Twenty-four randomized broad configurations are screened; eight advance to all five folds. Ten unique refinements are generated around the broad winner, and four advance to all five folds. Non-converged candidates remain visible but are ineligible for final selection.

In [0]:
lr_tuning_results, lr_full_rows, lr_screening_results = run_staged_search(
    "Logistic Regression", LOGISTIC_REGRESSION_BROAD_GRID, 8, 10, 4
)
display(lr_tuning_results)

## Tune Random Forest

Thirty-six randomized broad configurations are screened; ten advance to all five folds. Twelve unique refinements are generated around the broad winner, and five advance to all five folds.

In [0]:
rf_tuning_results, rf_full_rows, rf_screening_results = run_staged_search(
    "Random Forest", RANDOM_FOREST_BROAD_GRID, 10, 12, 5
)
display(rf_tuning_results)

## Tune XGBoost

Fifty-four randomized broad configurations are screened; twelve advance to all five folds. Eighteen unique refinements are generated around the broad winner, and six advance to all five folds.

In [0]:
xgb_tuning_results, xgb_full_rows, xgb_screening_results = run_staged_search(
    "XGBoost", XGBOOST_BROAD_GRID, 12, 18, 6
)
display(xgb_tuning_results)

## Rule-Based Final Candidate Selection

Final selection is applied across every promoted configuration—not one prematurely chosen row per algorithm. A configuration is eligible when it converged, its mean PR-AUC is within 0.02 of the best full-stage result, its PR-AUC standard deviation is no greater than 0.09, and its delayed-flight Recall at the common threshold rule is at least 0.50.

Eligible candidates are then ordered without arbitrary percentage weights: highest delayed F1, lowest Brier Score, highest Top-10% Lift, highest PR-AUC, higher interpretability, and lower training time. If the strict gates produce no candidate, the notebook records and applies a transparent fallback rather than silently hard-coding an algorithm.

In [0]:
all_full_rows = lr_full_rows + rf_full_rows + xgb_full_rows
best_pr_auc = max(row["PR_AUC"] for row in all_full_rows)
interpretability = {"Logistic Regression": 1.0, "Random Forest": 0.7, "XGBoost": 0.6}


def eligibility(row, recall_floor=MIN_FINAL_DELAY_RECALL, tolerance=PR_AUC_TOLERANCE):
    reasons = []
    if not row["CONVERGED"]:
        reasons.append("non-converged")
    if row["PR_AUC"] < best_pr_auc - tolerance:
        reasons.append("outside PR-AUC tolerance")
    if row["PR_AUC_STD"] > MAX_PR_AUC_STD:
        reasons.append("unstable PR-AUC")
    if row["DELAY_RECALL"] < recall_floor:
        reasons.append("below operational Recall floor")
    return not reasons, "; ".join(reasons) if reasons else "eligible"


def final_rule_key(row):
    return (
        -row["DELAY_F1"], row["BRIER_SCORE"], -row["TOP_10_LIFT"],
        -row["PR_AUC"], -row["INTERPRETABILITY_SCORE"], row["TRAINING_SECONDS"],
    )


scored_rows = []
for row in all_full_rows:
    candidate = dict(row)
    candidate["INTERPRETABILITY_SCORE"] = interpretability[candidate["MODEL"]]
    candidate["ELIGIBLE"], candidate["ELIGIBILITY_REASON"] = eligibility(candidate)
    scored_rows.append(candidate)

eligible_rows = [row for row in scored_rows if row["ELIGIBLE"]]
selection_fallback = "none"
if not eligible_rows:
    selection_fallback = "Recall floor relaxed to 0.40 and PR-AUC tolerance to 0.04"
    for row in scored_rows:
        row["ELIGIBLE"], row["ELIGIBILITY_REASON"] = eligibility(row, 0.40, 0.04)
    eligible_rows = [row for row in scored_rows if row["ELIGIBLE"]]
if not eligible_rows:
    selection_fallback = "No gates satisfied; all converged full-stage candidates ranked"
    eligible_rows = [row for row in scored_rows if row["CONVERGED"]]

eligible_rows = sorted(eligible_rows, key=final_rule_key)
for rank, row in enumerate(eligible_rows, start=1):
    row["SELECTION_RULE_RANK"] = rank

ranked_model_comparison = spark.createDataFrame(eligible_rows).orderBy("SELECTION_RULE_RANK")

representatives = []
for model_name in ["Logistic Regression", "Random Forest", "XGBoost"]:
    algorithm_rows = [row for row in scored_rows if row["MODEL"] == model_name]
    preferred = [row for row in algorithm_rows if row["ELIGIBLE"]] or algorithm_rows
    representatives.append(min(preferred, key=final_rule_key))
tuned_model_comparison = spark.createDataFrame(representatives)

best_logistic_regression = spark.createDataFrame([representatives[0]])
best_random_forest = spark.createDataFrame([representatives[1]])
best_xgboost = spark.createDataFrame([representatives[2]])

print(f"Best full-stage PR-AUC: {best_pr_auc:.4f}")
print(f"Selection fallback: {selection_fallback}")
display(ranked_model_comparison)
display(tuned_model_comparison)

## Tuned Confusion Matrices and Calibration Across Chronological Folds

Each algorithm's representative configuration is refitted separately within every chronological fold. Its stored operating threshold—not a hard-coded 0.50—is applied to the pooled out-of-fold probabilities. TP, FP, TN, and FN are reported explicitly, and calibration curves show whether predicted risks agree with observed delay frequencies.

In [0]:
best_parameter_rows = {
    "Logistic Regression": representatives[0],
    "Random Forest": representatives[1],
    "XGBoost": representatives[2],
}

for model_name, result_row in best_parameter_rows.items():
    params = ast.literal_eval(result_row["PARAMETERS"])
    threshold = float(result_row["OPERATING_THRESHOLD"])
    pooled_labels, pooled_probabilities = [], []
    for fold_index, fold in enumerate(LOCAL_CV_FOLDS):
        X_train, y_train = apply_imbalance_strategy(
            fold["X_train"], fold["y_train"], params["imbalance_strategy"],
            RANDOM_SEED + fold_index,
        )
        estimator = ESTIMATOR_BUILDERS[model_name](params, y_train)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            estimator.fit(X_train, y_train)
        pooled_labels.append(fold["y_validation"])
        pooled_probabilities.append(estimator.predict_proba(fold["X_validation"])[:, 1])
    labels = np.concatenate(pooled_labels)
    probabilities = np.concatenate(pooled_probabilities)
    predictions = (probabilities >= threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    print(f"{model_name} threshold={threshold:.4f}: TN={tn:,}, FP={fp:,}, FN={fn:,}, TP={tp:,}")
    display(spark.createDataFrame([
        (model_name, "TN", int(tn)), (model_name, "FP", int(fp)),
        (model_name, "FN", int(fn)), (model_name, "TP", int(tp)),
    ], ["MODEL", "OUTCOME", "COUNT"]))
    probability_true, probability_predicted = calibration_curve(
        labels, probabilities, n_bins=10, strategy="quantile"
    )
    plt.figure(figsize=(5, 4))
    plt.plot(probability_predicted, probability_true, marker="o", label=model_name)
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed delay rate")
    plt.title(f"Calibration: {model_name}")
    plt.legend()
    plt.show()

## Select the Candidate Final Model

The candidate is selected dynamically from all configurations that pass the documented convergence, ranking-performance, stability, and operational-Recall gates. The final ordering is rule based rather than a manually weighted score. Notebook 08 must still retrain this candidate and evaluate it on the untouched later holdout period before the project declares a definitive final model.

In [0]:
selected_model_row = ranked_model_comparison.first()
if selected_model_row is None:
    raise ValueError("No eligible tuned candidate was produced.")

SELECTED_MODEL_NAME = selected_model_row["MODEL"]
SELECTED_MODEL_PARAMETERS = ast.literal_eval(selected_model_row["PARAMETERS"])
SELECTED_MODEL_THRESHOLD = float(selected_model_row["OPERATING_THRESHOLD"])
# Compatibility alias for downstream notebooks that use this name.
SELECTION_THRESHOLD = SELECTED_MODEL_THRESHOLD
selected_model_summary = ranked_model_comparison.filter(
    F.col("SELECTION_RULE_RANK") == 1
).limit(1)

display(selected_model_summary)
print(f"Selected candidate: {SELECTED_MODEL_NAME}")
print(f"Selected parameters: {SELECTED_MODEL_PARAMETERS}")
print(f"Operating threshold: {SELECTED_MODEL_THRESHOLD:.4f}")
print("Rule: eligibility gates, then delayed F1, Brier Score, Top-10% Lift, PR-AUC, interpretability, and cost.")

In [0]:
test

## Candidate-Selection Interpretation

The selected row is the strongest eligible candidate under the documented rule; it is not selected solely because it has the highest Recall, ROC-AUC, or PR-AUC. The comparison first protects minority-class ranking quality and temporal stability, enforces an operational delayed-flight Recall floor, and then balances alert quality, calibration, top-risk usefulness, interpretability, and computational cost.

These values summarize five-fold chronological development performance. They support candidate selection but do not replace the untouched holdout evaluation in Notebook 08.

## Persist Modeling Checkpoints

This section saves the leakage-safe raw modeling tables, tuned-model comparison, preprocessing specification, and selected-candidate metadata required by downstream notebooks.

Feature hashing is no longer used. The preprocessing manifest records the ordered raw inputs and the scikit-learn `ColumnTransformer` design. Notebook 08 must fit a fresh preprocessor on its training period, transform validation and holdout data with that fitted transformer, reconstruct the selected estimator, and perform final calibration and holdout evaluation.


In [0]:
import json

checkpoint_tables = [
    (cfg.MODELING_TRAIN_HIST_TABLE, df_train_hist),
    (cfg.MODELING_VALIDATION_HIST_TABLE, df_validation_hist),
    (cfg.MODELING_TEST_HIST_TABLE, df_test_hist),
]

for table_name, dataframe in checkpoint_tables:
    row_count = dataframe.count()
    print(f"Saving {table_name}: {row_count:,} rows")
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

(
    tuned_model_comparison.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(cfg.TUNED_MODEL_COMPARISON_TABLE)
)

feature_manifest = {
    "model_input_columns": MODEL_INPUT_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "numerical_columns": NUMERICAL_COLUMNS,
    "target_column": TARGET_COLUMN,
    "preprocessing_library": "scikit-learn",
    "preprocessing_transformer": "ColumnTransformer",
    "categorical_transform": "most-frequent imputation plus OneHotEncoder(handle_unknown='ignore')",
    "numerical_transform": "median imputation plus StandardScaler(with_mean=False)",
    "fit_boundary": "fit on training only within each chronological fold",
    "selected_model_name": SELECTED_MODEL_NAME,
    "selected_model_parameters": SELECTED_MODEL_PARAMETERS,
}

candidate_selection = {
    "selected_model_name": SELECTED_MODEL_NAME,
    "selected_model_parameters": SELECTED_MODEL_PARAMETERS,
    "selection_source": "Five-fold chronological multi-criteria comparison",
    "selected_after_hyperparameter_tuning": True,
    "modeling_implementation": "standard_python",
    "cross_validation": "five expanding chronological folds",
    "class_imbalance_strategy": "natural, class-weighted, and training-only undersampled alternatives",
    "selection_policy": "rule-based eligibility gates followed by delayed F1, Brier Score, Top-10% Lift, PR AUC, interpretability, and computational cost",
    "selection_fallback": selection_fallback,
    "operating_threshold": SELECTED_MODEL_THRESHOLD,
}

dbutils.fs.put(
    cfg.MODEL_FEATURE_MANIFEST_PATH,
    json.dumps(feature_manifest, indent=4),
    overwrite=True,
)
dbutils.fs.put(
    cfg.CANDIDATE_SELECTION_PATH,
    json.dumps(candidate_selection, indent=4),
    overwrite=True,
)

print("Modeling checkpoints saved successfully.")
print(f"Preprocessing manifest: {cfg.MODEL_FEATURE_MANIFEST_PATH}")
print(f"Candidate selection: {cfg.CANDIDATE_SELECTION_PATH}")

In [0]:
candidate_selection_saved = json.loads(
    dbutils.fs.head(cfg.CANDIDATE_SELECTION_PATH, 10_000)
)
print(json.dumps(candidate_selection_saved, indent=4))

assert candidate_selection_saved["selected_model_name"] == SELECTED_MODEL_NAME
assert (
    candidate_selection_saved["selected_model_parameters"]
    == SELECTED_MODEL_PARAMETERS
)
print("Candidate-selection metadata verified.")
